# DAE-PINN

## 环境安装

本案例要求 **MindSpore >= 2.5.0** 版本以调用如下接口: *mindspore.jit, mindspore.jit_class, mindspore.data_sink*。具体请查看[MindSpore安装](https://www.mindspore.cn/install)。

此外，你需要安装 **MindScience >=0.1.0** 版本。如果当前环境还没有安装，请按照下列方式选择后端和版本进行安装。


In [ ]:
mindscience_version = "0.1.0"  # update if needed
# Comment out the following code if you are using NPU.
!pip uninstall -y mindscience-ascend
!pip install mindscience-ascend==$mindscience_version

# NPU Uncomment if needed.
# !pip uninstall -y mindscience-ascend
# !pip install mindscience-ascend==$mindscience_version

## 概述

* **电力网络动态安全评估需求** ：随着电力网络中分布式能源资源的整合、市场自由化以及复杂通信和控制算法的采用，电力网络的运行条件和潜在故障场景变得更加多样化，影响其安全性。为了评估电力网络的动态安全性，需要模拟其在面对单一故障时的动态响应，这需要求解一组非线性微分代数方程（DAE），而传统显式积分方案在求解 DAE 时会失败，商业求解器计算成本高、内存需求大，限制了动态安全评估的在线部署。
* **深度学习在科学和工程领域的潜力与挑战** ：尽管深度学习在计算机视觉和自然语言处理等领域取得了巨大成功，但在学习科学和工程动态系统方面应用有限，因为数据收集成本高昂，且大多数传统深度学习方法在数据量有限的情况下缺乏鲁棒性和泛化能力。

## 工作原理

* DAE-PINNs 框架结合了隐式龙格 - 库塔时间步进方案（专为求解 DAE 设计）和物理信息神经网络（PINN）。在时间步进过程中，假设已积分至 $(t_n, y_n, z_n)$，目标是推进至 $(t_{n+1}, y_{n+1}, z_{n+1})$，应用隐式龙格 - 库塔方案后，得到一系列方程，包括内部阶段的更新公式和最终状态的更新公式。
* 通过惩罚方法强制神经网络满足 DAE 作为近似硬约束。在训练过程中，将 DAE 的残差作为损失函数的一部分，使得网络在学习过程中不仅拟合数据，还能满足物理定律所描述的 DAE 方程，从而将物理信息融入到神经网络的学习过程中。

## 方法细节

* **问题设置** ：DAE 以半显式形式给出，包括动态状态 y 和代数变量 z，以及描述微分方程的 f 和代数方程的 g。假设 f 和 g 具有足够高的可微性，并且 DAE 的索引为 1，即雅可比矩阵 g_z 的逆存在且在精确解附近有界，这使得代数方程在局部有唯一解 $z = G(y)$，从而 DAE 可以转化为普通微分方程系统。
* **网络结构** ：与标准的 PINN 类似，DAE-PINNs 通常由输入层、多个隐藏层和输出层组成。输入层接收时间和动态状态等信息，隐藏层通过非线性激活函数进行特征提取和转换，输出层预测代数变量的值。
* **损失函数** ：损失函数由两部分组成，一部分是数据损失，用于拟合初始条件、边界条件等已知点的数据；另一部分是物理损失，即 DAE 的残差损失，通过自动微分计算网络输出对时间和状态变量的导数，代入 DAE 方程得到残差，并将其作为物理损失的一部分。通过优化这两个部分的损失函数，使得网络既能拟合数据，又能满足物理方程。

![model](./images/model.png)

[DAE-PINN](https://arxiv.org/abs/2109.04304)的网络结构如上图。

与传统神经网络不同，DAE-PINN 在网络结构中融入了物理信息，通过构造特定的损失函数，使网络在学习过程中不仅拟合数据，还能满足物理定律所描述的 DAE 方程，从而提高了模型的准确性和泛化能力。整体的网络架构如上，分为两个网络分别处理动态状态和代数状态，网络的输入是包括时间信息和动态状态信息，网络的输出是对动态状态和代数状态的预测值，具体来说，会输出动态状态 y 和代数变量 z 的预测结果，如在电力网络案例中，输出动态状态和代数变量的预测以实现对电力网络动态行为的模拟。网络支持使用`fnn`、`attention`、`conv1d`3种backbone。`fnn`为多层感知机网络，`attention`为采用类似transformer attention形式的FFN网络，`conv1d`为使用了`Conv1D`的FFN网络。

## 准备环节

实践前，确保已经正确安装最新版本的MindSpore与mindscience。如果没有，可以通过：

* [MindSpore安装页面](https://www.mindspore.cn/install) 安装MindSpore。
* [mindscience安装页面](https://gitee.com/mindspore/mindscience) 安装mindscience。


## DAE-PINN实现

DAE-PINN实现分为以下5个步骤：

1. 配置网络与训练参数
2. 数据集制作与加载
3. 模型构建
4. 模型训练
5. 结果可视化


In [ ]:
import time

from mindspore import ops, jit
from mindspore import context
from mindspore.experimental import optim
import numpy as np

from mindscience.utils import load_yaml_config

from src.utils import dotdict
from src.model import three_bus_PN
from src.data import get_dataset
from src.trainer import DaeTrainer


In [ ]:
context.set_context(mode=context.PYNATIVE_MODE, device_target='Ascend', device_id=1)

## 配置网络与训练参数

从配置文件中读取模型相关参数（model）、数据相关参数（data）、优化器相关参数（optimizer），设置模型初始化、特征维度、子网类型等参数。


In [ ]:
config = load_yaml_config('./configs/config.yaml')
model_params, data_params, optim_params, ode_params, summary_params = config[
    'model'], config['data'], config['optimizer'], config['ode'], config['summary']

dynamic = dotdict()
dynamic.num_IRK_stages = model_params['num_IRK_stages']
dynamic.state_dim = 4
dynamic.activation = model_params['dyn_activation']
dynamic.initializer = "Glorot normal"
dynamic.dropout_rate = 0
dynamic.batch_normalization = None if model_params['dyn_bn'] == "no-bn" else model_params['dyn_bn']
dynamic.layer_normalization = None if model_params['dyn_ln'] == "no-ln" else model_params['dyn_ln']
dynamic.type = model_params['dyn_type']

if model_params['unstacked']:
    dim_out = dynamic.state_dim * (dynamic.num_IRK_stages + 1)
else:
    dim_out = dynamic.num_IRK_stages + 1

if model_params['use_input_layer']:
    dynamic.layer_size = [dynamic.state_dim * 5] + \
        [model_params['dyn_width']] * model_params['dyn_depth'] + [dim_out]
else:
    dynamic.layer_size = [dynamic.state_dim] + \
        [model_params['dyn_width']] * model_params['dyn_depth'] + [dim_out]

algebraic = dotdict()
algebraic.num_IRK_stages = model_params['num_IRK_stages']
dim_out_alg = algebraic.num_IRK_stages + 1
algebraic.layer_size = [dynamic.state_dim] + \
    [model_params['alg_width']] * model_params['alg_depth'] + [dim_out_alg]
algebraic.activation = model_params['alg_activation']
algebraic.initializer = "Glorot normal"
algebraic.dropout_rate = 0
algebraic.batch_normalization = None if model_params['alg_bn'] == "no-bn" else model_params['alg_bn']
algebraic.layer_normalization = None if model_params['alg_ln'] == "no-ln" else model_params['alg_ln']
algebraic.type = model_params['alg_type']

## 数据集制作与加载

数据集下载地址：

该文件包含6000条HyperCube数据集。


In [ ]:
train_dataset, test_dataset, val_dataset = get_dataset(data_params)

## 模型构建


In [ ]:

net = three_bus_PN(
    dynamic,
    algebraic,
    use_input_layer=model_params['use_input_layer'],
    stacked=not model_params['unstacked'],
)

## 损失函数与优化器

损失函数采用了MSE函数。优化器选用了Adam，学习率调度采用了ReduceLROnPlateau。


In [ ]:
num_IRK_stages = model_params['num_IRK_stages']
# collecting RK data
data_dir = data_params['data_dir']
irk_data = np.loadtxt(os.path.join(data_dir, 'IRK_weights', f'Butcher_IRK{num_IRK_stages}.txt'), ndmin=2, dtype=np.float32)
IRK_weights = np.reshape(
    irk_data[0:num_IRK_stages**2+num_IRK_stages], (num_IRK_stages+1, num_IRK_stages))
IRK_weights = Tensor(IRK_weights)
IRK_times = irk_data[num_IRK_stages**2 + num_IRK_stages:]
trainer = DaeTrainer(net, IRK_weights=IRK_weights, IRK_times=IRK_times, h=ode_params['h'],
                     dyn_weight=model_params['dyn_weight'], alg_weight=model_params['alg_weight'])

optimizer = optim.Adam(net.trainable_params(), lr=optim_params['lr'])
scheduler_type = optim_params['scheduler_type']
use_scheduler = optim_params['use_scheduler']
if use_scheduler:
    if scheduler_type == "plateau":
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            patience=optim_params['patience'],
            factor=optim_params['factor'],
        )
    elif scheduler_type == "step":
        scheduler = optim.lr_scheduler.StepLR(
            optimizer, step_size=optim_params['patience'], gamma=optim_params['factor']
        )
    else:
        scheduler = None
else:
    scheduler = None


## 训练函数

使用**MindSpore>= 2.5.0**的版本，可以使用函数式编程范式训练神经网络，单步训练函数使用jit装饰。


In [ ]:

def forward_fn(x):
    loss = trainer.get_loss(x)
    return loss

grad_fn = ops.value_and_grad(
    forward_fn, None, optimizer.parameters, has_aux=False)

@jit
def train_step(x):
    loss, grads = grad_fn(x)
    optimizer(grads)
    return loss

## 模型训练

模型训练过程中边训练边推理。用户可以直接加载测试数据集，每训练n个epoch后输出一次测试集上的推理精度。


In [ ]:
test_interval = summary_params['test_interval']

for epoch in range(1, 1 + optim_params['epochs']):
    # train
    time_beg = time.time()
    net.set_train(True)
    loss_val = []
    for data, in train_dataset:
        step_train_loss = train_step(data)
        loss_val.append(step_train_loss.numpy())
    time_end = time.time()

    print(
        f"epoch: {epoch} train loss: {np.mean(loss_val)} epoch time: {time_end-time_beg:.3f}s")
    if use_scheduler:
        if scheduler_type == "plateau":
            net.set_train(False)
            loss_val = trainer.get_loss(val_dataset)
            scheduler.step(loss_val)
        else:
            scheduler.step()
    # test
    if epoch % test_interval == 0:
        net.set_train(False)
        loss_test = trainer.get_loss(test_dataset)
        print(f'test loss: {loss_test}')


## 结果可视化

模型训练的loss下降曲线如下：

![model](images/loss.png)

可以看到训练3W轮之后，训练和测试loss均下降到5e-3左右。

网络对于4个动态和1个代数变量预测的L2相对损失如下图:

![动态变量0误差图](./images/L2relative_error_0.png)
![动态变量1误差图](./images/L2relative_error_1.png)
![动态变量2误差图](./images/L2relative_error_2.png)
![动态变量3误差图](./images/L2relative_error_3.png)
![代数变量误差图](./images/L2relative_error_4.png)
